In [1]:
import instructor
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
import time

In [2]:
# Define the structure you want
class Step(BaseModel):
    """Describes a step in the AR Training Experience"""
    
    name: str = Field(
        description="A short, descriptive identifier for the step"
    )
    title: str = Field(
        description="A clear and concise task description, to be displayed"
    )
    description: str = Field(
        description="Detailed verbal instructions for the user, guiding them through the task to be performed"
    )

class ARLab(BaseModel):
    """A list of AR training steps"""
    steps: List[Step] = Field(
        description="Complete list of all the steps in the AR training experience"
    )

In [13]:
# strings identifying which api providers to link the client to
# defaults to cheapest new model, but model can be specified below
providers = [
    "openai/gpt-5-nano",               # [0] :
    "google/gemini-2.5-flash",         # [1] : Gemini models do not support Union types in the pydantic models
    "anthropic/claude-haiku-3"         # [2] :
]

# Model selection, what models are available for each provider, and token costs per 1MTok
models = {
    providers[0] : [# OpenAI: Input  : Cached : Output
        "gpt-5-nano", # [0] : $00.05 : $00.01 : $00.40
        "gpt-5-mini", # [1] : $00.25 : $00.03 : $02.00
        "04-mini",    # [2] : $01.10 : $00.28 : $04.40
        "gpt-5.1",    # [3] : $01.25 : $00.13 : $10.00
        "gpt-5.2"     # [4] : $01.75 : $00.18 : $14.00
    ],
    providers[1] : [# Gemini            : Input  : >200kT/P : Cached : >200kT/P : Store1MT/h : Output : >200kT/P
        "gemini-2.5-flash",       # [0] : $00.30 :  $=====  : $00.03 :  $=====  :   $01.00   : $02.50 :  $=====
        "gemini-2.5-pro",         # [1] : $01.25 :  $02.50  : $00.125:  $00.25  :   $04.50   : $10.00 :  $15.00
        "gemini-3-flash-preview", # [2] : $00.50 :  $=====  : $00.05 :  $=====  :   $01.00   : $03.00 :  $=====
        "gemini-3-pro-preview"    # [3] : $02.00 :  $04.00  : $00.20 :  $00.40  :   $04.50   : $12.00 :  $18.00
    ],
    providers[2] : [# Anthropic    : Input  : Cached : 5mStore : 1hStore : Output 
        "claude-haiku-3",    # [0] : $00.25 : $00.03 : $00.30  : $00.50  : $01.25
        "claude-haiku-3-5",  # [1] : $00.80 : $00.08 : $01.00  : $01.60  : $04.00
        "claude-haiku-4-5",  # [2] : $01.00 : $00.10 : $01.25  : $02.00  : $05.00
        "claude-sonnet-4-5", # [3] : $03.00 : $00.30 : $03.75  : $06.00  : $15.00
       #"claude-opus-4-5"    # [4] : $05.00 : $00.50 : $06.25  : $10.00  : $25.00   ### WAAAAY to expensive
    ]
}

In [4]:
# Load the environment file with our api keys set
# create a .env file in the same folder, put your api key environment variables there
# Don't forget to put the .env in the gitignore so you don't publish your api keys!
load_dotenv()

True

In [5]:
# API Key test. You don't need to run this every time
# If anything is wrong with your api key, this should fail now
openai_client = instructor.from_provider(providers[0])
google_client = instructor.from_provider(providers[1])
anthropic_client = instructor.from_provider(providers[2])

In [6]:
# Prompt outlinining the models task. this is a simple, quick, rough draft prompt, and can be improtved
prompt = """You are an AI agent tasked with generating step by step instructional scenarios for XR training experiences.
Each step is made of the following elements:
Name, Title, and Description.

Generate a training scenario by creating a sequence of steps to accomplish the objective:
Learn how to use the basic functions of a Carvey CNC machine to be able to use it for general operation.
The target audience is:
Students and staff at a university that have varying levels of familiarity with this technology.
The Activity that will be done to accomplish the objective is:
Use the Carvey to carve a stylized letter from wood.

Ensure each step is relevant, accurate, and practical for achieving the training objective within an XR environment,
and try to make the experience 6 steps long
"""

In [7]:
def test_client(client, prompt):
    # start timer
    start_time = time.time()

    # Extract structured data
    # Old method, without access to raw response
    # lab = client.create(
    # New method, which gives us access to token usage
    lab, completion = client.create_with_completion(
        response_model=ARLab,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_retries=3
    )

    # end timer
    end_time = time.time()

    # total time taken
    elapsed_time = end_time - start_time
    
    # Return the lab, completion, and time taken.
    return lab, completion, elapsed_time

In [15]:
# Quick price calculation

token_costs = {    #In    Out
    providers[0] : [0.05, 0.40], # openAI gpt5n
    providers[1] : [0.30, 2.50], # google g2.5f
    providers[2] : [0.25, 1.25]  # anthropic ch3
}

def calculate_price(completion, ic, oc):
    input_tokens = completion.usage.prompt_tokens
    output_tokens = completion.usage.completion_tokens

    input_cost = input_tokens * ic / 1000000
    output_cost = output_tokens * oc / 1000000
    total_cost = input_cost + output_cost

    # Return input tokens, output tokens, and total cost
    return input_tokens, output_tokens, total_cost

In [ ]:
### NOTE: if this keeps failing on a specific provider,
###       try testing only that client in a new cell to save on token cost

# Test Each Client
for idx, provider in enumerate(providers):
    print(f"Testing Provider : {provider}\n")
    client = instructor.from_provider(provider)

    try:
        lab, completion, e_time = test_client(client, prompt)
    except Exception as e:
        print(f"Prompt failed with error : {str(e)}\n")
    else:
        print(f"Prompt completed in {e_time}s\n")

        in_c_t, out_c_t = token_costs[provider]
        in_t, out_t, cost = calculate_price(completion, in_c_t, out_c_t)
        print(f"Resources Used :")
        print(f"Input Tokens  = {in_t}")
        print(f"Output Tokens = {out_t}")
        print(f"Total Cost    = ${cost}\n")
        
        for i, step in enumerate(lab.steps):
            print(f"Step {i} : {step.name}\nTitle : {step.title}\nDescription : {step.description}\n\n")

Testing Provider : openai/gpt-5-nano

Prompt completed in 52.131019830703735s

Resources Used :
Input Tokens  = 364
Output Tokens = 4500
Total Cost    = $0.0018181999999999999

Step 0 : safety_briefing
Title : Safety briefing, XR orientation, and objectives
Description : Welcome to the Carvey XR training. You will learn the basic functions required to safely operate the Carvey CNC and to perform a simple wood carving task. In this step you will review personal protective equipment (PPE), workspace safety, and machine hazards. Identify the emergency stop and power switch, dust collection, and wasteboard boundaries. Demonstrate a quick safety check in XR by confirming you understand the hazards, know how to pause or stop the machine, and can access the control panel.

Learning outcomes:
- Recognize safety requirements for CNC operation
- Locate and understand the purpose of major machine controls
- Demonstrate a basic safety check before powering on the machine


Step 1 : machine_overvie

AttributeError: 'GenerateContentResponse' object has no attribute 'usage'